# Proximal Policy Optimization


A2C — developed in [02-policy-gradients.html](./02-policy-gradients.html) — advances on REINFORCE by replacing Monte Carlo returns with one-step TD advantages and introducing a shared actor-critic architecture. It works, but it carries two structural weaknesses. First, the bias-variance tradeoff in advantage estimation is dictated entirely by the rollout length: short rollouts mean high bias (the critic dominates), long rollouts mean high variance (Monte Carlo noise dominates). There is no principled knob to tune. Second, every gradient step on the collected batch moves $\pi_\theta$ in an unrestricted direction. A single bad step can destroy learned behavior and send training into a recovery spiral.

This notebook introduces two ideas that address these weaknesses directly. **Generalized Advantage Estimation** (GAE) adds a parameter $\lambda \in [0, 1]$ that blends one-step TD and full Monte Carlo returns via an exponential weighting over $n$-step advantages. Setting $\lambda$ near 0 recovers TD(0); setting it to 1 recovers the full Monte Carlo estimate. In practice $\lambda = 0.95$ provides a robust working point that accepts modest bias to achieve substantial variance reduction. **Proximal Policy Optimization** (PPO) addresses the step-size problem not with a hard constraint (as in TRPO) but with a clipped surrogate objective: the importance ratio $\rho_t = \pi_\theta / \pi_{\theta_{\text{old}}}$ is clamped to $[1 - \epsilon, 1 + \epsilon]$, which removes the gradient signal whenever the new policy has already moved far from the old one.

PPO is the algorithm behind RLHF for large language models. The clipped surrogate objective is reused verbatim in GRPO — the only differences are that the trajectory is a token sequence, the advantage is computed from group rewards rather than GAE, and the behavior policy is the reference model. Understanding PPO mechanically is the bridge to the final notebook in this series on LLM alignment.

**Prerequisites:** [02-policy-gradients.html](./02-policy-gradients.html). We use PyTorch for neural networks and `gymnasium` for CartPole-v1.


## Generalized Advantage Estimation


Recall from the previous notebook that A2C uses the one-step TD advantage:

$$\hat{A}_t^{\text{TD}(0)} = \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t),$$

which is low-variance but biased whenever $V$ is not perfect. At the other extreme, the Monte Carlo advantage

$$\hat{A}_t^{\text{MC}} = G_t - V(s_t) = \sum_{k=0}^{T-t-1} \gamma^k r_{t+k} - V(s_t)$$

is unbiased but high-variance, since $G_t$ accumulates noise across many future steps. GAE unifies both via the $n$-step return. Define:

$$G_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k r_{t+k} + \gamma^n V(s_{t+n}).$$

The corresponding $n$-step advantage $\hat{A}_t^{(n)} = G_t^{(n)} - V(s_t)$ can be written as a telescoping sum of TD errors:

$$\hat{A}_t^{(n)} = \sum_{k=0}^{n-1} \gamma^k \delta_{t+k}.$$

GAE is the exponentially-weighted average of all $n$-step advantages, with decay parameter $\lambda$:

$$\hat{A}_t^{\text{GAE}(\gamma, \lambda)} = (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} \hat{A}_t^{(n)} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}.$$

The two boundary cases are illuminating. When $\lambda = 0$, only the $l = 0$ term survives and we recover $\hat{A}_t = \delta_t$ — the one-step TD advantage. When $\lambda = 1$, the sum is $\sum_{l=0}^{\infty} \gamma^l \delta_{t+l} = G_t - V(s_t)$ — the full Monte Carlo advantage. So $\lambda$ smoothly interpolates between the two extremes, with larger $\lambda$ admitting more variance in exchange for less bias.

In practice the rollout has finite horizon $T$, so we truncate the sum:

$$\hat{A}_t^{\text{GAE}} = \sum_{l=0}^{T-t-1} (\gamma \lambda)^l \delta_{t+l}.$$

This has an efficient recursive form. Scanning backwards from $t = T-1$ to $t = 0$:

$$\hat{A}_t = \delta_t + \gamma \lambda \hat{A}_{t+1},$$

with $\hat{A}_{T} = 0.$ The GAE return used as the critic target is $G_t^{\text{GAE}} = \hat{A}_t + V(s_t).$


**Setup.** We import all dependencies and fix seeds for reproducibility:


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")

torch.manual_seed(0)
np.random.seed(0)


**GAE computation.** We implement the backward scan that computes advantages and returns for an entire rollout:


In [ ]:
def compute_gae(rewards, values, dones, next_value, gamma=0.99, lam=0.95):
    """
    Compute GAE advantages and returns.

    Args:
        rewards:    list of T rewards
        values:     tensor of T state values V(s_t)
        dones:      list of T done flags
        next_value: scalar V(s_T) for bootstrapping
        gamma, lam: discount and GAE lambda

    Returns:
        advantages: tensor of shape (T,)
        returns:    tensor of shape (T,)  — GAE returns = advantages + values
    """
    T = len(rewards)
    advantages = torch.zeros(T)
    gae = 0.0

    for t in reversed(range(T)):                             # <1>
        next_val = next_value if t == T - 1 else values[t + 1].item()
        mask = 1.0 - float(dones[t])                         # <2>
        delta = rewards[t] + gamma * next_val * mask - values[t].item()
        gae = delta + gamma * lam * mask * gae               # <3>
        advantages[t] = gae

    returns = advantages + values                            # <4>
    return advantages, returns


1. We scan backwards because each $\hat{A}_t$ depends on future TD errors $\delta_{t+1}, \delta_{t+2}, \ldots$ via the recurrence $\hat{A}_t = \delta_t + \gamma\lambda \hat{A}_{t+1}.$
2. The done mask zeros out the bootstrap from the next state at episode boundaries, preventing value leakage across episodes.
3. Accumulate the exponentially-weighted sum using the recurrence. At a terminal state the mask is 0, so `gae` resets for the new episode.
4. The GAE return $G_t^{\text{GAE}} = \hat{A}_t + V(s_t)$ is used as the critic target during the optimization phase.


**Verifying the boundary cases.** We construct a simple 4-step rollout with known values and confirm that $\lambda = 0$ recovers TD(0) advantages and $\lambda = 1$ recovers Monte Carlo advantages:


In [ ]:
gamma = 0.99
rewards = [1.0, 1.0, 1.0, 1.0]
values  = torch.tensor([0.5, 0.6, 0.7, 0.8])
dones   = [False, False, False, False]
next_v  = 0.9

# lambda=0 should give one-step TD deltas
adv_td, _ = compute_gae(rewards, values, dones, next_v, gamma=gamma, lam=0.0)
delta_manual = torch.tensor([
    rewards[t] + gamma * (values[t+1].item() if t < 3 else next_v) - values[t].item()
    for t in range(4)
])
print("lambda=0 (TD):  ", adv_td.numpy().round(6))
print("manual TD delta:", delta_manual.numpy().round(6))
print("match:", torch.allclose(adv_td, delta_manual, atol=1e-6))

# lambda=1 should give MC advantages  G_t - V(s_t)
adv_mc, _ = compute_gae(rewards, values, dones, next_v, gamma=gamma, lam=1.0)
G = [0.0] * 4
G[3] = rewards[3] + gamma * next_v
for t in reversed(range(3)):
    G[t] = rewards[t] + gamma * G[t+1]
mc_manual = torch.tensor([G[t] - values[t].item() for t in range(4)])
print("\nlambda=1 (MC):  ", adv_mc.numpy().round(6))
print("manual MC adv:  ", mc_manual.numpy().round(6))
print("match:", torch.allclose(adv_mc, mc_manual, atol=1e-6))


:::{.callout-note}
The $\lambda$ parameter controls the effective horizon of the advantage estimate. Small $\lambda$ (near 0) gives low-variance but biased TD advantages; large $\lambda$ (near 1) gives unbiased but high-variance MC advantages. In practice $\lambda = 0.95$ is a robust default that accepts modest bias for substantial variance reduction.

:::


## Trust Regions and PPO


A2C performs gradient ascent on the current policy objective using data collected from that same policy. Once we take a gradient step, the data is stale — the new policy is no longer the one that generated the transitions. In principle, we could reuse the data by correcting for the distribution shift via importance sampling: replace $\mathbb{E}_{a \sim \pi_\theta}[\ldots]$ with $\mathbb{E}_{a \sim \pi_{\theta_{\text{old}}}}[\rho_t \cdot \ldots]$, where

$$\rho_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}$$

is the **importance sampling ratio.** This gives the surrogate objective:

$$L^{\text{IS}}(\theta) = \mathbb{E}_t\!\left[\rho_t(\theta)\, \hat{A}_t\right].$$

The problem is that $\rho_t$ is unbounded. When $\pi_\theta$ diverges from $\pi_{\theta_{\text{old}}}$, the correction can become very large or very small, destabilizing training. **TRPO** handles this by adding an explicit KL constraint:

$$\max_\theta \; L^{\text{IS}}(\theta) \quad \text{subject to} \quad D_\text{KL}(\pi_{\theta_{\text{old}}} \| \pi_\theta) \leq \delta.$$

Solving this constrained problem requires conjugate gradients and a line search — correct but expensive. **PPO** replaces the constraint with a clipped objective:

$$\boxed{L^{\text{CLIP}}(\theta) = \mathbb{E}_t\!\left[\min\!\left(\rho_t(\theta)\,\hat{A}_t,\; \text{clip}(\rho_t(\theta),\, 1-\epsilon,\, 1+\epsilon)\,\hat{A}_t\right)\right].}$$

The mechanics of the clip are most clearly seen by case analysis. Let $\rho = \rho_t(\theta)$ and $A = \hat{A}_t$:

| Sign of $\hat{A}_t$ | Value of $\rho_t$ | Effect |
|---|---|---|
| $\hat{A}_t > 0$ (good action) | $\rho_t < 1 + \epsilon$ | Unclipped: gradient pushes $\pi_\theta(a_t)$ up |
| $\hat{A}_t > 0$ (good action) | $\rho_t > 1 + \epsilon$ | Clipped: no gradient, cap enforced |
| $\hat{A}_t < 0$ (bad action) | $\rho_t > 1 - \epsilon$ | Unclipped: gradient pushes $\pi_\theta(a_t)$ down |
| $\hat{A}_t < 0$ (bad action) | $\rho_t < 1 - \epsilon$ | Clipped: no gradient, floor enforced |

The `min` ensures we always take the more conservative (lower) bound. When $\hat{A}_t > 0$ the unclipped objective would happily increase $\rho_t$ without bound, but the `min` with the clipped version caps its contribution once $\rho_t > 1 + \epsilon.$ The symmetric argument applies when $\hat{A}_t < 0.$ Together, the clip + min enforce conservative updates in both directions — without solving a constrained optimization problem.

The full PPO objective adds a value function loss and an entropy bonus:

$$L^{\text{PPO}}(\theta) = L^{\text{CLIP}}(\theta) - c_1 L^{\text{VF}}(\theta) + c_2 H[\pi_\theta(\cdot \mid s_t)],$$

where $L^{\text{VF}} = \frac{1}{T}\sum_t (V_\phi(s_t) - G_t^{\text{GAE}})^2$ trains the critic, and $H[\pi_\theta]$ is an entropy bonus that discourages premature collapse to a deterministic policy. Typical values are $c_1 = 0.5$ and $c_2 = 0.01.$


:::{.callout-important}
The clipped surrogate $L^{\text{CLIP}}$ is reused verbatim in GRPO. In GRPO the importance ratio is computed at the sequence level ($\pi_\theta(y \mid x) / \pi_{\theta_{\text{old}}}(y \mid x)$) rather than per-action, and the advantage is the normalized group reward rather than GAE — but the clipping mechanism is identical.

:::


## PPO Implementation


**Networks.** We redefine `PolicyNet` and `ValueNet` here for completeness, following the same architecture as in [02-policy-gradients.html](./02-policy-gradients.html). The key addition on `PolicyNet` is an `evaluate` method that computes log probabilities and entropy for a batch of state-action pairs — needed during the off-policy update phase:


In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, obs_dim=4, hidden_dim=64, n_actions=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, n_actions),
        )

    def forward(self, x):
        return F.softmax(self.net(x), dim=-1)

    def act(self, obs):
        """Sample action; return (action int, log_prob scalar tensor)."""
        obs_t = torch.FloatTensor(obs).unsqueeze(0)
        probs = self(obs_t)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)

    def evaluate(self, states, actions):                     # <1>
        """Evaluate a batch; return log_probs and per-sample entropy."""
        probs = self(states)
        dist = torch.distributions.Categorical(probs)
        log_probs = dist.log_prob(actions)
        entropy = dist.entropy()
        return log_probs, entropy


class ValueNet(nn.Module):
    def __init__(self, obs_dim=4, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


1. `evaluate` is the workhorse of the PPO update: it re-evaluates $\log \pi_\theta(a_t \mid s_t)$ under the current (updated) policy for a stored batch of $(s_t, a_t)$ pairs, enabling multiple gradient epochs on the same rollout.


**PPO agent.** The `PPO` class encapsulates rollout collection and the multi-epoch minibatch update:


In [ ]:
class PPO:
    def __init__(
        self,
        obs_dim=4,
        n_actions=2,
        hidden_dim=64,
        lr=3e-4,
        gamma=0.99,
        lam=0.95,
        clip_eps=0.2,
        c1=0.5,
        c2=0.01,
        n_epochs=4,      # <1>
        batch_size=64,   # <2>
    ):
        self.policy    = PolicyNet(obs_dim, hidden_dim, n_actions)
        self.value_net = ValueNet(obs_dim, hidden_dim)
        self.optimizer = torch.optim.Adam(
            list(self.policy.parameters()) + list(self.value_net.parameters()),
            lr=lr,
        )
        self.gamma     = gamma
        self.lam       = lam
        self.clip_eps  = clip_eps
        self.c1        = c1
        self.c2        = c2
        self.n_epochs  = n_epochs
        self.batch_size = batch_size

    def collect_rollout(self, env, rollout_steps=2048):
        """Collect rollout_steps transitions. Returns a batch dict."""
        states, actions, rewards, dones, log_probs_old, values = [], [], [], [], [], []
        obs, _ = env.reset()
        episode_returns = []
        current_return  = 0.0

        for _ in range(rollout_steps):
            action, log_prob = self.policy.act(obs)
            with torch.no_grad():
                value = self.value_net(
                    torch.FloatTensor(obs).unsqueeze(0)
                ).item()

            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            states.append(obs)
            actions.append(action)
            rewards.append(reward)
            dones.append(done)
            log_probs_old.append(log_prob.item())
            values.append(value)

            current_return += reward
            obs = next_obs
            if done:
                episode_returns.append(current_return)
                current_return = 0.0
                obs, _ = env.reset()

        # Bootstrap value for the last state
        with torch.no_grad():
            next_value = self.value_net(
                torch.FloatTensor(obs).unsqueeze(0)
            ).item()

        values_t = torch.FloatTensor(values)
        advantages, returns = compute_gae(
            rewards, values_t, dones, next_value, self.gamma, self.lam
        )

        return {
            "states":          torch.FloatTensor(np.array(states)),
            "actions":         torch.LongTensor(actions),
            "log_probs_old":   torch.FloatTensor(log_probs_old),
            "advantages":      advantages,
            "returns":         returns,
            "episode_returns": episode_returns,
        }

    def update(self, batch):
        """Run n_epochs of minibatch PPO updates on the collected rollout."""
        states        = batch["states"]
        actions       = batch["actions"]
        log_probs_old = batch["log_probs_old"]
        advantages    = batch["advantages"]
        returns       = batch["returns"]

        # Normalize advantages over the full rollout                 # <3>
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        T = len(states)
        metrics = {
            "policy_loss": [], "value_loss": [], "entropy": [],
            "approx_kl": [], "clip_fraction": [],
        }

        for _ in range(self.n_epochs):
            indices = torch.randperm(T)                              # <4>

            for start in range(0, T, self.batch_size):
                idx = indices[start:start + self.batch_size]

                mb_states        = states[idx]
                mb_actions       = actions[idx]
                mb_log_probs_old = log_probs_old[idx]
                mb_advantages    = advantages[idx]
                mb_returns       = returns[idx]

                log_probs_new, entropy = self.policy.evaluate(mb_states, mb_actions)

                # Importance ratio in log-space for numerical stability
                ratio = (log_probs_new - mb_log_probs_old).exp()     # <5>

                # Clipped surrogate objective
                surr1 = ratio * mb_advantages
                surr2 = ratio.clamp(
                    1 - self.clip_eps, 1 + self.clip_eps
                ) * mb_advantages
                policy_loss = -torch.min(surr1, surr2).mean()

                # Critic loss
                values_pred = self.value_net(mb_states)
                value_loss  = F.mse_loss(values_pred, mb_returns)

                loss = policy_loss + self.c1 * value_loss - self.c2 * entropy.mean()

                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(self.policy.parameters()) + list(self.value_net.parameters()),
                    0.5,
                )
                self.optimizer.step()

                # Diagnostics (no grad needed)
                with torch.no_grad():
                    approx_kl  = ((mb_log_probs_old - log_probs_new) ** 2).mean() / 2  # <6>
                    clip_frac  = ((ratio - 1).abs() > self.clip_eps).float().mean()
                    metrics["policy_loss"].append(policy_loss.item())
                    metrics["value_loss"].append(value_loss.item())
                    metrics["entropy"].append(entropy.mean().item())
                    metrics["approx_kl"].append(approx_kl.item())
                    metrics["clip_fraction"].append(clip_frac.item())

        return {k: float(np.mean(v)) for k, v in metrics.items()}


1. PPO's key advantage over A2C: the same rollout is reused for $K$ gradient epochs. Each environment interaction contributes to $K$ updates instead of one, substantially improving sample efficiency.
2. Small minibatches within each epoch add stochasticity and decorrelate the gradient estimates, which can help with generalization.
3. Normalizing advantages over the entire rollout before updates is a standard PPO implementation detail. It stabilizes training by ensuring the gradient magnitude does not depend on the scale of rewards.
4. Shuffling the rollout each epoch ensures that minibatches are uncorrelated across epochs, preventing the network from overfitting to a particular ordering.
5. We compute the importance ratio in log-space as $\exp(\log \pi_{\text{new}} - \log \pi_{\text{old}})$ for numerical stability — direct division of probabilities can underflow or overflow with many actions.
6. An approximate KL divergence $D_\text{KL}(\pi_{\text{old}} \| \pi_{\text{new}}) \approx \frac{1}{2}\mathbb{E}[(\log \rho_t)^2]$ derived from a second-order Taylor expansion of the KL. This is a cheap diagnostic: if it consistently exceeds 0.02–0.05, the policy is updating too aggressively.


**Training loop.** We wrap rollout collection and updates into a single training function:


In [ ]:
def train_ppo(n_updates=100, rollout_steps=2048, **ppo_kwargs):
    env   = gym.make("CartPole-v1")
    agent = PPO(**ppo_kwargs)

    all_episode_returns = []
    update_metrics      = []

    for update in range(n_updates):
        batch   = agent.collect_rollout(env, rollout_steps)
        metrics = agent.update(batch)

        all_episode_returns.extend(batch["episode_returns"])
        update_metrics.append(metrics)

        if (update + 1) % 10 == 0:
            recent = all_episode_returns[-20:] if all_episode_returns else [0]
            print(
                f"Update {update+1:3d}  "
                f"ep_return={np.mean(recent):.1f}  "
                f"clip_frac={metrics['clip_fraction']:.3f}  "
                f"approx_kl={metrics['approx_kl']:.4f}  "
                f"entropy={metrics['entropy']:.3f}"
            )

    env.close()
    return agent, all_episode_returns, update_metrics


**Training.** We run PPO for 100 update steps, each collecting 2048 environment transitions:


In [ ]:
agent_ppo, returns_ppo, metrics_ppo = train_ppo(n_updates=100, rollout_steps=2048)


**Figure.** Four training diagnostics over the 100 update steps:


In [ ]:
#| code-fold: true
def smooth(x, window=20):
    if len(x) < window:
        return np.array(x)
    return np.convolve(x, np.ones(window) / window, mode="valid")

keys   = ["policy_loss", "clip_fraction", "approx_kl", "entropy"]
labels = ["Policy loss", "Clip fraction", "Approx KL", "Entropy"]
colors = ["steelblue", "darkorange", "seagreen", "mediumpurple"]

fig, axes = plt.subplots(2, 2, figsize=(10, 6))

for ax, key, label, color in zip(axes.flat, keys, labels, colors):
    vals = [m[key] for m in metrics_ppo]
    ax.plot(vals, alpha=0.35, color=color, linewidth=0.9)
    ax.plot(
        np.arange(len(smooth(vals)) - 1 + (len(vals) - len(smooth(vals)) + 1 - 1),
                  len(vals)),
        smooth(vals),
        color=color, linewidth=2,
    )
    ax.set_xlabel("Update")
    ax.set_title(label)
    ax.grid(linestyle="dotted", alpha=0.5)

# Overlay episode returns on policy loss panel with a twin axis
ax0 = axes[0, 0]
ax0_r = ax0.twinx()
ep_arr = np.array(returns_ppo)
ax0_r.plot(ep_arr, alpha=0.12, color="tomato", linewidth=0.5)
if len(ep_arr) >= 20:
    ax0_r.plot(
        np.arange(19, len(ep_arr)),
        smooth(ep_arr, window=20),
        color="tomato", linewidth=1.5, label="Ep return (right)",
    )
ax0_r.axhline(475, color="tomato", linestyle="--", linewidth=1.0, alpha=0.6)
ax0_r.set_ylabel("Episode return", color="tomato")
ax0_r.tick_params(axis="y", labelcolor="tomato")

plt.suptitle("PPO training diagnostics — CartPole-v1", fontsize=12)
plt.tight_layout()
plt.show()


**Figure.** PPO training diagnostics over 100 update steps. (**top-left**) Policy loss (negative clipped surrogate) with episode return on the right axis — as returns approach 475, the policy loss magnitude stabilizes near zero, indicating the policy is no longer being pushed aggressively. (**top-right**) Clip fraction: the fraction of timesteps where the importance ratio exceeded $[1-\epsilon, 1+\epsilon]$; a healthy value is 10–30%. (**bottom-left**) Approximate KL divergence, measuring how much the policy shifts per update; values consistently below 0.02 indicate the clip is working. (**bottom-right**) Policy entropy, decreasing slowly as the policy becomes more decisive over training.


## PPO Diagnostics


Training diagnostics are not just monitoring tools — they are the primary signal for detecting misconfigured hyperparameters before the learning curve collapses. The table below summarizes what each metric indicates and when to be concerned:

| Metric | Healthy range | Warning sign |
|---|---|---|
| Clip fraction | 10–30% | >50%: steps too large; ~0%: clip is irrelevant |
| Approx KL | < 0.02 | >0.05: policy changing too fast per update |
| Policy entropy | Decreasing slowly | Rapid collapse: policy too greedy, exploration lost |
| Value loss | Decreasing overall | Plateau with poor returns: critic not fitting |

A complementary scalar summary of critic quality is the **explained variance**:

$$\text{EV} = 1 - \frac{\operatorname{Var}(G_t - V_\phi(s_t))}{\operatorname{Var}(G_t)}.$$

When $\text{EV} \approx 1$, the critic captures most of the variability in the return — it is a strong baseline that substantially reduces the variance of advantage estimates. When $\text{EV} \approx 0$, the critic is no better than a constant predictor and the advantages carry the full return variance. Negative values indicate the critic is actively misleading.


**Explained variance.** We compute $\text{EV}$ from the last collected rollout and display a diagnostic summary:


In [ ]:
# Re-collect a fresh rollout to evaluate the trained agent
env_eval = gym.make("CartPole-v1")
eval_batch = agent_ppo.collect_rollout(env_eval, rollout_steps=2048)
env_eval.close()

G  = eval_batch["returns"].numpy()
A  = eval_batch["advantages"].numpy()

# Explained variance: 1 - Var(G - V) / Var(G)
# G = A + V  =>  G - V = A (un-normalized advantages before update)
var_G   = np.var(G)
var_res = np.var(A)  # residuals before advantage normalization
ev      = 1.0 - var_res / (var_G + 1e-8)

print(f"Explained variance : {ev:.4f}")
print(f"Var(G)             : {var_G:.4f}")
print(f"Var(G - V)         : {var_res:.4f}")
print(f"Mean episode return: {np.mean(eval_batch['episode_returns']):.1f}")


:::{.callout-note}
These same diagnostics appear in GRPO training: mean KL from the reference model, clipped fraction, reward standard deviation, and response entropy. The RL foundations are exactly the same — only the environment (CartPole vs. LLM generation) differs.

:::


## Ablations


We run two ablations to verify the design choices in PPO. The first varies the clip parameter $\epsilon$; the second varies the GAE parameter $\lambda.$ Each ablation holds all other hyperparameters fixed and trains for 60 updates to keep total wall time manageable.


**Effect of clip $\epsilon$.** We train with $\epsilon \in \{0.1, 0.2, 0.3\}$ and compare learning curves:


In [ ]:
clip_eps_values = [0.1, 0.2, 0.3]
results_clip = {}

for eps in clip_eps_values:
    torch.manual_seed(0)
    np.random.seed(0)
    print(f"clip_eps={eps}")
    _, ep_returns, _ = train_ppo(n_updates=60, rollout_steps=2048, clip_eps=eps)
    results_clip[eps] = ep_returns


**Figure.** Learning curves for different clip values:


In [ ]:
#| code-fold: true
colors_eps = {0.1: "steelblue", 0.2: "seagreen", 0.3: "darkorange"}

fig, ax = plt.subplots(figsize=(7, 3.5))
for eps, ep_ret in results_clip.items():
    arr = np.array(ep_ret)
    ax.plot(arr, alpha=0.15, color=colors_eps[eps], linewidth=0.6)
    if len(arr) >= 20:
        ax.plot(
            np.arange(19, len(arr)),
            smooth(arr, window=20),
            color=colors_eps[eps], linewidth=2,
            label=f"$\\epsilon={eps}$",
        )
ax.axhline(475, color="tomato", linestyle="--", linewidth=1.2, label="Solved (475)")
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title("PPO: effect of clip $\\epsilon$")
ax.legend(loc="upper left", fontsize=9)
ax.grid(linestyle="dotted", alpha=0.5)
plt.tight_layout()
plt.show()


**Figure.** Learning curves for $\epsilon \in \{0.1, 0.2, 0.3\}$. Too-small $\epsilon$ (0.1) over-constrains each update and slows convergence. Too-large $\epsilon$ (0.3) allows aggressive steps that can cause instability. The default $\epsilon = 0.2$ offers a robust balance.


**Effect of GAE $\lambda$.** We train with $\lambda \in \{0.0, 0.5, 0.95, 1.0\}$ to observe the bias-variance tradeoff in practice:


In [ ]:
#| code-fold: true
lam_values = [0.0, 0.5, 0.95, 1.0]
results_lam = {}

for lam in lam_values:
    torch.manual_seed(0)
    np.random.seed(0)
    print(f"lam={lam}")
    _, ep_returns, _ = train_ppo(n_updates=60, rollout_steps=2048, lam=lam)
    results_lam[lam] = ep_returns

colors_lam = {0.0: "steelblue", 0.5: "darkorange", 0.95: "seagreen", 1.0: "mediumpurple"}

fig, ax = plt.subplots(figsize=(7, 3.5))
for lam, ep_ret in results_lam.items():
    arr = np.array(ep_ret)
    ax.plot(arr, alpha=0.12, color=colors_lam[lam], linewidth=0.5)
    if len(arr) >= 20:
        ax.plot(
            np.arange(19, len(arr)),
            smooth(arr, window=20),
            color=colors_lam[lam], linewidth=2,
            label=f"$\\lambda={lam}$",
        )
ax.axhline(475, color="tomato", linestyle="--", linewidth=1.2, label="Solved (475)")
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title("PPO: effect of GAE $\\lambda$")
ax.legend(loc="upper left", fontsize=9)
ax.grid(linestyle="dotted", alpha=0.5)
plt.tight_layout()
plt.show()


**Figure.** Learning curves for $\lambda \in \{0.0, 0.5, 0.95, 1.0\}$. $\lambda = 0$ (pure TD) tends to converge slowly because the biased advantages mislead the policy early in training when the critic is still imprecise. $\lambda = 1$ (pure MC) carries high variance that can prevent stable convergence. The intermediate values $\lambda = 0.95$ and $\lambda = 0.5$ offer the best of both worlds, with $\lambda = 0.95$ typically converging fastest on CartPole.


## Appendix: PPO Pseudocode Reference


The pseudocode below summarizes the complete PPO algorithm. The next notebook in this series maps each component directly to its LLM alignment counterpart (GRPO/RLHF): the rollout phase becomes prompt-conditioned generation, GAE returns become normalized group rewards, and the behavior policy becomes the reference model.

```python
# PPO (Schulman et al., 2017) — pseudocode
# Hyperparameters: gamma, lam, clip_eps, c1, c2, K (epochs), M (batch size), T (rollout)

initialize policy pi_theta, critic V_phi

for each update:

    # === ROLLOUT PHASE ===
    collect T transitions (s_t, a_t, r_t, done_t) using pi_theta
    record log_pi_theta_old(a_t | s_t) for each step

    # Compute GAE advantages (backward scan)
    delta_t = r_t + gamma * V(s_{t+1}) * (1 - done_t) - V(s_t)
    A_t     = delta_t + (gamma * lam) * A_{t+1}  (with A_T = 0)
    G_t     = A_t + V(s_t)                       (GAE returns for critic)

    # Normalize advantages
    A_t <- (A_t - mean(A_t)) / (std(A_t) + 1e-8)

    # === OPTIMIZATION PHASE ===
    for k = 1, ..., K:                           # K epochs on same data
        shuffle rollout indices
        for each minibatch of size M:
            ratio   = exp(log_pi_theta(a|s) - log_pi_theta_old(a|s))
            L_CLIP  = min(ratio * A, clip(ratio, 1-eps, 1+eps) * A).mean()
            L_VF    = mse(V_phi(s), G)
            L_H     = entropy(pi_theta(. | s)).mean()
            loss    = -L_CLIP + c1 * L_VF - c2 * L_H
            update theta, phi via gradient descent on loss
            clip gradients to max norm 0.5

    # behavior policy stays fixed within the K epochs;
    # log_pi_theta_old is refreshed only at the next rollout phase
```


---


■
